# PDX Synthetic SLM Dispatch Fair Comparison (Dual Tesla T4)
## Evaluating Isolated Step Dispatch vs Accumulating ReAct with Gold Plan Held Constant

This benchmark executes a strictly controlled empirical test of:
- **Condition A: `goldplan_react`** (Single accumulating multi-turn conversation)
- **Condition B: `goldplan_isolated`** (Fresh messages per step, only parent receipts)

### Strict Experimental Controls:
1. **Same Gold Plan**: Both conditions receive identical step breakdown and instructions.
2. **Same Full Context**: Zero head/tail cropping (no 150-char truncation).
3. **Same Honest Tools**: No gold fallbacks; tools operate strictly on model-provided arguments.
4. **Same JSON Output Schema**: Strict JSON output required from both conditions.
5. **Same Generation Parameters**: `temperature=0.0`, `do_sample=False`, `max_new_tokens=256` (repetition penalty removed).
6. **Same Scorer**: Identical terminal answer evaluation (`score_answer`).
7. **Synthetic Fixtures**: All tasks are synthetic benchmark fixtures (not real GAIA/RULER/SWE).


In [ ]:
import subprocess
import os
import sys

print("=== 1. Hardware Preflight Check ===")
gpu_query = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    text=True, capture_output=True, check=True
).stdout.strip().splitlines()

print('Detected GPUs:', gpu_query)
assert len(gpu_query) == 2, f'Dual T4 benchmark requires two GPUs, found {gpu_query}'
assert all('T4' in item.upper() for item in gpu_query), f'Dual T4 required; found: {gpu_query}'
print("Hardware check passed: Dual Tesla T4 present.")


In [ ]:
print("=== 2. Environment Setup ===")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "transformers>=4.50.0", "accelerate", "pdx-artifact-engine==0.3.0a9"], check=True)
import torch
import transformers
import pdx_artifact_engine

print("PyTorch Version:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| GPU Count:", torch.cuda.device_count())
print("Transformers Version:", transformers.__version__)
print("PDX Artifact Engine Installed (engine_invocations will be recorded in manifest).")


In [ ]:
# === 3. Honest Benchmark Tools Environment ===
"""Honest Benchmark Tool Environment for Fair SLM Evaluation.

Strictly follows Section 4 of FAIR_COMPARISON_PLAN.md:
- No gold fallbacks.
- Operates only on explicit model-provided arguments.
- Rejects missing or invalid parameters with TOOL_ERROR.
"""

from __future__ import annotations

import base64
import hashlib
import json
import math
import re
from typing import Any, Dict, Optional


FIXTURE_DB: Dict[str, str] = {
    "user_819": json.dumps({"encrypted_token": "UERYX0FVVEhfOTkyMQ==", "cipher": "base64"}),
    "cl_alpha": json.dumps({"latencies_ms": [120, 150, 135, 160, 145]}),
    "SKU_9021": json.dumps({"stock": 45, "target": 100, "batch_size": 20}),
    "pkg_linux_x64": json.dumps({"cas_hash": "a8f9c1e0992", "status": "verified", "merkle_root": "ROOT_MERKLE_OK"}),
    "PRODOCUX_SPEC_V2": json.dumps({"author": "Ada Lovelace", "status": "approved"}),
    "asset_list": json.dumps(["asset_1012", "asset_4402", "asset_9918", "asset_3301", "asset_9918"]),
    "telemetry_services": json.dumps({"auth": 45, "db": 120, "image_embedder": 890, "cache": 5}),
    "endpoint_export": json.dumps({"required_role": "admin"}),
    "pkg_c": json.dumps({"prerequisite": "pkg_b", "root_prerequisite": "pkg_a"}),
    "server_port": json.dumps({"host": "0.0.0.0", "port": 8080}),
}


def tool_calc(args: Dict[str, Any]) -> str:
    expr = args.get("expression") or args.get("expr")
    if not expr or not isinstance(expr, str) or not expr.strip():
        return "TOOL_ERROR: missing expression"
    
    expr = expr.strip()
    # Check for safe characters: digits, basic operators, parens, decimal, sqrt, whitespace, comma
    safe_pattern = re.compile(r"^[0-9\+\-\*\/\%\(\)\.\,\s]|sqrt$")
    cleaned = expr.replace("sqrt", "").replace("math.", "")
    if not all(c in "0123456789+-*/%()., \t\n" for c in cleaned):
        return "TOOL_ERROR: unsafe expression"
    
    try:
        # Evaluate in restricted environment with math.sqrt and ceil
        safe_env = {
            "__builtins__": None,
            "sqrt": math.sqrt,
            "ceil": math.ceil,
            "floor": math.floor,
        }
        res = eval(expr, safe_env)
        return str(res)
    except Exception as exc:
        return f"TOOL_ERROR: eval failed: {exc}"


def tool_grep(args: Dict[str, Any], context: str) -> str:
    query = args.get("query")
    if not query or not isinstance(query, str) or not query.strip():
        return "TOOL_ERROR: missing query"
    
    q = query.strip()
    lines = context.splitlines()
    matches = [line for line in lines if q in line]
    if not matches:
        return "NO_MATCH"
    return "\n".join(matches[:5])


def tool_db_lookup(args: Dict[str, Any]) -> str:
    key = args.get("key") or args.get("query") or args.get("id") or args.get("name")
    if not key or not isinstance(key, str) or not key.strip():
        return "TOOL_ERROR: missing key"
    
    k = key.strip()
    # Exact key match only - no author heuristic fallback
    if k in FIXTURE_DB:
        return FIXTURE_DB[k]
    return "NOT_FOUND"


def tool_crypto_transform(args: Dict[str, Any]) -> str:
    op = args.get("op")
    data = args.get("data")
    if not op:
        return "TOOL_ERROR: missing op"
    if data is None:
        return "TOOL_ERROR: missing data"
    
    data_str = str(data).strip()
    if op == "b64decode":
        try:
            # Handle potential padding issues
            pad = len(data_str) % 4
            if pad != 0:
                data_str += "=" * (4 - pad)
            decoded = base64.b64decode(data_str.encode("utf-8")).decode("utf-8", errors="replace")
            return decoded
        except Exception as exc:
            return f"TOOL_ERROR: decode failed: {exc}"
    elif op == "sha256_prefix":
        digest = hashlib.sha256(data_str.encode("utf-8")).hexdigest()
        return digest[:8]
    else:
        return f"TOOL_ERROR: unsupported op '{op}'"


def tool_json_filter(args: Dict[str, Any]) -> str:
    json_input = args.get("json") or args.get("data")
    predicate = args.get("predicate") or args.get("query") or args.get("key")
    if not json_input:
        return "TOOL_ERROR: missing json"
    
    try:
        parsed = json.loads(json_input) if isinstance(json_input, str) else json_input
    except Exception as exc:
        return f"TOOL_ERROR: invalid json: {exc}"
    
    if not predicate:
        return json.dumps(parsed)
    
    pred = str(predicate).strip()
    if isinstance(parsed, dict):
        if pred in parsed:
            return str(parsed[pred])
        return "NOT_FOUND"
    elif isinstance(parsed, list):
        filtered = [item for item in parsed if pred in str(item)]
        return json.dumps(filtered)
    return str(parsed)


def execute_benchmark_tool(tool_name: str, args: Dict[str, Any], context: str = "") -> str:
    """Entrypoint for tool execution. Never falls back to gold answers."""
    if not isinstance(args, dict):
        return "TOOL_ERROR: args must be a dictionary"
    
    if tool_name == "calc":
        return tool_calc(args)
    elif tool_name == "grep":
        return tool_grep(args, context)
    elif tool_name == "db_lookup":
        return tool_db_lookup(args)
    elif tool_name == "crypto_transform":
        return tool_crypto_transform(args)
    elif tool_name == "json_filter":
        return tool_json_filter(args)
    elif tool_name == "run_python":
        return "TOOL_ERROR: run_python is not available"
    else:
        return f"TOOL_ERROR: unknown tool '{tool_name}'"


In [ ]:
# === 4. Fair Answer Scoring Module ===
"""Honest Answer Scoring for Fair SLM Evaluation.

Strictly follows Section 5 of FAIR_COMPARISON_PLAN.md:
- Numeric equivalence (relative error <= 1e-3, rounded equality at 1 decimal, handles % and $).
- Exact match after casefold and punctuation stripping.
- Rejects sentence dumps (forces extraction of target needle/token).
- Never searches intermediate tool traces.
"""

from __future__ import annotations

import math
import re
from typing import Optional, Tuple


def normalize_text(text: str) -> str:
    """Normalize text by stripping markdown formatting, quotes, trailing periods, and casefolding."""
    if not text:
        return ""
    s = text.strip()
    # Strip markdown code fences / backticks
    s = re.sub(r"^```[a-zA-Z]*\n?", "", s)
    s = re.sub(r"\n?```$", "", s)
    s = s.strip("`'\" \t\r\n")
    # Strip trailing punctuation if it's a period or comma at the end of a sentence
    s = re.sub(r"[\.,;:]+$", "", s)
    return s.strip().lower()


def extract_first_number(text: str) -> Optional[float]:
    """Extract first numeric value from string, handling % and $."""
    if not text:
        return None
    # Clean standard currency and percent symbols for numeric parsing
    cleaned = text.replace("$", "").replace(",", "")
    match = re.search(r"[-+]?(?:\d+\.?\d*|\.\d+)", cleaned)
    if match:
        try:
            return float(match.group(0))
        except ValueError:
            return None
    return None


def is_numeric_expected(expected: str) -> Optional[float]:
    """Check if expected string is cleanly numeric (scalar, currency, or percentage).
    
    Returns parsed float if numeric, or None if string/identifier/code.
    """
    if not expected:
        return None
    s = str(expected).strip().replace("$", "").replace("%", "").replace(",", "").strip()
    if re.fullmatch(r"[-+]?(?:\d+(?:\.\d+)?|\.\d+)", s):
        try:
            return float(s)
        except ValueError:
            return None
    return None


def score_answer(predicted: str, expected: str, suite: str = "") -> Tuple[bool, str]:
    """Score predicted answer against expected answer according to fair protocol.
    
    Returns:
        (passed: bool, reason: str)
    """
    if predicted is None:
        return False, "null_prediction"
    
    pred_str = str(predicted).strip()
    exp_str = str(expected).strip()
    
    if not pred_str:
        return False, "empty_prediction"
    
    # 1. Exact match after normalize
    norm_pred = normalize_text(pred_str)
    norm_exp = normalize_text(exp_str)
    
    if norm_pred == norm_exp:
        return True, "exact_match"
    
    # 2. Numeric comparison ONLY when expected answer is genuinely numeric
    exp_num = is_numeric_expected(exp_str)
    if exp_num is not None:
        pred_num = extract_first_number(pred_str)
        if pred_num is not None:
            # Check if the predicted text is predominantly numeric (not a paragraph dump)
            words_in_pred = pred_str.split()
            is_numeric_suite = any(s in suite for s in ["calc", "gsm", "finance", "gaia_01", "gaia_03", "gaia_07", "gaia_08", "gaia_13"])
            if len(words_in_pred) <= 6 or is_numeric_suite:
                # Check relative and absolute tolerance
                if math.isclose(pred_num, exp_num, rel_tol=1e-3, abs_tol=1e-2):
                    return True, "numeric_close"
                if round(pred_num, 1) == round(exp_num, 1):
                    return True, "numeric_rounded"
    
    # 3. String / Token / Code comparison
    # Do NOT allow substring match when predicted is a full sentence dumping the context
    # Casefold exact match failed above.
    return False, f"mismatch: expected '{norm_exp}', got '{norm_pred}'"


In [ ]:
# === 5. Standardized Benchmark Tasks Fixtures ===
"""Comprehensive Two-Tier Benchmark Suite for PDX Phase 0 Large-Scale Evaluation.

Contains 76 standardized tasks across two distinct difficulty tiers:
=== Tier 1: Level 1 Baseline (40 tasks) ===
1. gaia_l1 (15 tasks): Single-step tool use, document audit, numerical calculation.
2. ruler_l1 (15 tasks): Single-needle retrieval across 4k to 32k contexts.
3. swe_l1 (10 tasks): Basic Python bug fixes and schema validations.

=== Tier 2: Level 2 Advanced (36 tasks) ===
4. dag_chain (8 tasks): Multi-step DAG tool chaining with strict upstream dependencies.
5. gaia_l2 (8 tasks): GAIA Level 2 multi-tool tabular and multi-constraint reasoning.
6. swe_algo (8 tasks): Algorithmic Python code repair with unit test verification.
7. gsm8k_hard (6 tasks): Multi-step compound arithmetic requiring tool calculator execution.
8. ruler_multihop (6 tasks): Multi-hop needle-in-a-haystack context retrieval (6k-12k tokens).
"""

from __future__ import annotations

import json
from dataclasses import dataclass
from typing import Any, Dict, List, Optional


@dataclass
class BenchmarkTask:
    task_id: str
    suite: str
    tier: str  # "level_1" or "level_2"
    prompt: str
    context: str
    tools: List[Dict[str, Any]]
    expected_answer: str
    pdx_steps: List[Dict[str, Any]]
    max_steps: int = 5
    difficulty: str = "medium"

    def get_full_gold_steps(self) -> List[Dict[str, Any]]:
        steps = [dict(s) for s in self.pdx_steps]
        if not any(s.get("id") == "step_final" for s in steps):
            prev_ids = [s["id"] for s in steps]
            steps.append({
                "id": "step_final",
                "kind": "final",
                "instruction": "Output the task's final answer only.",
                "depends_on": prev_ids,
            })
        return steps

    def to_pdx_plan(self) -> Dict[str, Any]:
        return {
            "schema_version": "pdx_execution_plan_v1",
            "request_id": f"phase0_{self.suite}_{self.task_id}",
            "producer": {"type": "benchmark", "name": f"pdx-phase0-{self.suite}"},
            "intent": {
                "summary": self.prompt,
                "suite": self.suite,
                "tier": self.tier,
                "task_id": self.task_id,
                "max_steps": self.max_steps,
            },
            "environment": {
                "context_length": len(self.context),
                "available_tools": [t.get("name") for t in self.tools],
            },
            "steps": self.get_full_gold_steps(),
            "verification": [
                {
                    "id": "final_answer_match",
                    "check": "exact_or_substring",
                    "expected": self.expected_answer,
                }
            ],
            "policies": {
                "timeout_seconds": 600,
                "max_retries": 1,
                "default_approval_required": False,
            },
        }


def _noise(lines: int, text: str = "System telemetry OK. Cluster health nominal. P99 latency within SLA bounds.\n") -> str:
    return text * lines


BENCHMARK_TASKS: List[BenchmarkTask] = [
    # =========================================================================
    # TIER 1: LEVEL 1 BASELINE (40 tasks)
    # =========================================================================

    # 1. GAIA Level 1 (15 tasks)
    BenchmarkTask("gaia_01_revenue_calc", "gaia_l1", "level_1", "Find total revenue in Q3 2024 and compute year-over-year growth rate compared to Q3 2023.", "Financial Report: 2023 Q3 Revenue was $120.0M. 2024 Q3 Revenue was $156.0M.", [{"name": "calc"}], "30%", [
        {"id": "step_1_calc", "kind": "tool", "tool": "calc", "instruction": "Compute ((156.0 - 120.0) / 120.0) * 100", "depends_on": []}
    ]),
    BenchmarkTask("gaia_02_file_audit", "gaia_l1", "level_1", "Examine the system manifest and output the single invalid file name if any.", "Manifest: file_a.pdf: valid; file_b.docx: INVALID; file_c.json: valid", [{"name": "json_filter"}], "file_b.docx", [
        {"id": "step_1_audit", "kind": "transform", "instruction": "Examine manifest and identify invalid file name", "depends_on": []}
    ]),
    BenchmarkTask("gaia_03_date_calc", "gaia_l1", "level_1", "Calculate the day difference between deployment on 2024-05-10 and verification on 2024-05-24.", "Project log: start=2024-05-10, verified=2024-05-24.", [{"name": "calc"}], "14", [
        {"id": "step_1_diff", "kind": "tool", "tool": "calc", "instruction": "Compute 24 - 10", "depends_on": []}
    ]),
    BenchmarkTask("gaia_04_author_lookup", "gaia_l1", "level_1", "Find the author of spec document PRODOCUX_SPEC_V2.", "Registry: PRODOCUX_SPEC_V2 authored by Ada Lovelace, approved by QA team.", [{"name": "db_lookup"}], "Ada Lovelace", [
        {"id": "step_1_author", "kind": "transform", "instruction": "Identify author of PRODOCUX_SPEC_V2 from context", "depends_on": []}
    ]),
    BenchmarkTask("gaia_05_version_filter", "gaia_l1", "level_1", "Find the latest release version among v1.2, v2.0.1, v1.9, and v2.1.0-rc1.", "Release tags: v1.2, v2.0.1, v1.9, v2.1.0-rc1.", [{"name": "json_filter"}], "v2.1.0-rc1", [
        {"id": "step_1_filter", "kind": "transform", "instruction": "Find highest semver among candidate versions", "depends_on": []}
    ]),
    BenchmarkTask("gaia_06_port_inspect", "gaia_l1", "level_1", "What is the default HTTP service port specified for ProDocuX adapter?", "Config: server.host=0.0.0.0, server.port=8080, mode=production.", [{"name": "db_lookup"}], "8080", [
        {"id": "step_1_port", "kind": "transform", "instruction": "Extract server.port value from config", "depends_on": []}
    ]),
    BenchmarkTask("gaia_07_status_count", "gaia_l1", "level_1", "Count the total number of failed jobs in the batch execution log.", "Jobs: job_1=success, job_2=failed, job_3=failed, job_4=success, job_5=failed.", [{"name": "calc"}], "3", [
        {"id": "step_1_count", "kind": "transform", "instruction": "Count failed jobs in log list", "depends_on": []}
    ]),
    BenchmarkTask("gaia_08_token_estimate", "gaia_l1", "level_1", "Estimate token cost for 500k words with a multiplier of 1.3 tokens/word.", "Formula: tokens = 500000 * 1.3.", [{"name": "calc"}], "650000", [
        {"id": "step_1_estimate", "kind": "tool", "tool": "calc", "instruction": "Compute 500000 * 1.3", "depends_on": []}
    ]),
    BenchmarkTask("gaia_09_duplicate_finder", "gaia_l1", "level_1", "Identify the duplicated asset ID present in the asset registration list.", "Assets: asset_1012, asset_4402, asset_9918, asset_3301, asset_9918.", [{"name": "db_lookup"}], "asset_9918", [
        {"id": "step_1_dup", "kind": "transform", "instruction": "Identify duplicated asset identifier", "depends_on": []}
    ]),
    BenchmarkTask("gaia_10_missing_key", "gaia_l1", "level_1", "Identify the missing required configuration key among host, port, and database_uri.", "Supplied config: {'host': 'localhost', 'port': 5432}.", [{"name": "json_filter"}], "database_uri", [
        {"id": "step_1_missing", "kind": "transform", "instruction": "Identify which required key is missing", "depends_on": []}
    ]),
    BenchmarkTask("gaia_11_max_latency", "gaia_l1", "level_1", "Identify the service with the highest latency from the telemetry table.", "Latency table: auth=45ms, db=120ms, image_embedder=890ms, cache=5ms.", [{"name": "db_lookup"}], "image_embedder", [
        {"id": "step_1_max", "kind": "transform", "instruction": "Identify service name with maximum latency", "depends_on": []}
    ]),
    BenchmarkTask("gaia_12_permission_check", "gaia_l1", "level_1", "What user role is required to access the /export endpoint?", "Security policy: /view requires user, /export requires admin, /ping is public.", [{"name": "db_lookup"}], "admin", [
        {"id": "step_1_role", "kind": "transform", "instruction": "Extract role requirement for /export endpoint", "depends_on": []}
    ]),
    BenchmarkTask("gaia_13_file_size_sum", "gaia_l1", "level_1", "Sum the total size of attachments in MB: file1=12MB, file2=45MB, file3=18MB.", "Attachments: 12MB, 45MB, 18MB.", [{"name": "calc"}], "75", [
        {"id": "step_1_sum", "kind": "tool", "tool": "calc", "instruction": "Compute 12 + 45 + 18", "depends_on": []}
    ]),
    BenchmarkTask("gaia_14_hash_verify", "gaia_l1", "level_1", "Identify the MD5 prefix recorded for artifact release package.", "Checksum table: package_zip=a8f9c1e0992..., status=verified.", [{"name": "crypto_transform"}], "a8f9c1", [
        {"id": "step_1_hash", "kind": "transform", "instruction": "Extract first 6 hex characters of package_zip checksum", "depends_on": []}
    ]),
    BenchmarkTask("gaia_15_dependency_order", "gaia_l1", "level_1", "Resolve the top-level prerequisite package for pkg_c if pkg_c depends on pkg_b, which depends on pkg_a.", "Dependency chain: pkg_c -> pkg_b -> pkg_a.", [{"name": "db_lookup"}], "pkg_a", [
        {"id": "step_1_dep", "kind": "transform", "instruction": "Resolve root prerequisite package name", "depends_on": []}
    ]),

    # 2. RULER Level 1 (15 tasks)
    BenchmarkTask("ruler_01_needle_4k", "ruler_l1", "level_1", "What is the secret deployment token assigned to server node-melbourne-4?", _noise(60) + "server node-melbourne-4 token is TRN2-SECRET-MEL-9921.\n" + _noise(60), [{"name": "grep"}], "TRN2-SECRET-MEL-9921", [
        {"id": "step_1_grep", "kind": "tool", "tool": "grep", "instruction": "Search for token assigned to node-melbourne-4", "depends_on": []}
    ]),
    BenchmarkTask("ruler_02_needle_4k_ip", "ruler_l1", "level_1", "What is the internal IP address assigned to node-brazil-01?", _noise(70) + "Internal IP for node-brazil-01 is 10.240.12.88.\n" + _noise(60), [{"name": "grep"}], "10.240.12.88", [
        {"id": "step_1_grep", "kind": "tool", "tool": "grep", "instruction": "Search for IP address of node-brazil-01", "depends_on": []}
    ]),
    BenchmarkTask("ruler_03_needle_8k_key", "ruler_l1", "level_1", "Find the active production API key for payment gateway.", _noise(120) + "The active production key is sk-live-prod-88716.\n" + _noise(120), [{"name": "grep"}], "sk-live-prod-88716", [
        {"id": "step_1_grep", "kind": "tool", "tool": "grep", "instruction": "Search for production API key for payment gateway", "depends_on": []}
    ]),
    BenchmarkTask("ruler_04_needle_8k_city", "ruler_l1", "level_1", "Which AWS region is designated for Capacity Block reservations?", _noise(130) + "Designated Capacity Block region is ap-southeast-4 (Melbourne).\n" + _noise(110), [{"name": "grep"}], "ap-southeast-4", [
        {"id": "step_1_grep", "kind": "tool", "tool": "grep", "instruction": "Search for designated Capacity Block AWS region", "depends_on": []}
    ]),
    BenchmarkTask("ruler_05_needle_16k_uuid", "ruler_l1", "level_1", "Extract the run UUID of the failed pipeline job.", _noise(180) + "Pipeline failed: run_id=b4901f4a-9912-4c22-86ee-23ad4123.\n" + _noise(180), [{"name": "grep"}], "b4901f4a-9912-4c22-86ee-23ad4123", [
        {"id": "step_1_grep", "kind": "tool", "tool": "grep", "instruction": "Search for run_id of failed pipeline", "depends_on": []}
    ]),
    BenchmarkTask("ruler_06_needle_16k_user", "ruler_l1", "level_1", "Which user account was flagged for multiple failed login attempts?", _noise(190) + "Security alert: suspicious activity on account user_victor_99.\n" + _noise(190), [{"name": "grep"}], "user_victor_99", [
        {"id": "step_1_grep", "kind": "tool", "tool": "grep", "instruction": "Search for suspicious activity flagged user account", "depends_on": []}
    ]),
    BenchmarkTask("ruler_07_needle_32k_pass", "ruler_l1", "level_1", "What is the recovery passphrase stored in the disaster backup record?", _noise(260) + "Emergency recovery passphrase: crimson-falcon-orbit-992.\n" + _noise(260), [{"name": "grep"}], "crimson-falcon-orbit-992", [
        {"id": "step_1_grep", "kind": "tool", "tool": "grep", "instruction": "Search for emergency recovery passphrase", "depends_on": []}
    ]),
    BenchmarkTask("ruler_08_multihop_4k", "ruler_l1", "level_1", "Track user session user_882: find login IP and final exported file ID.", _noise(60) + "user_882 login from 192.168.10.42.\n" + _noise(50) + "user_882 exported artifact_doc_7719.pdf.\n" + _noise(40), [{"name": "grep"}], "192.168.10.42", [
        {"id": "step_1_grep", "kind": "tool", "tool": "grep", "instruction": "Search user_882 login IP address", "depends_on": []}
    ]),
    BenchmarkTask("ruler_09_multihop_8k", "ruler_l1", "level_1", "Find the author and commit hash that introduced the hotfix.", _noise(110) + "Commit 7a88b19 merged by Alice Zhang (hotfix: memory leak).\n" + _noise(120), [{"name": "grep"}], "Alice Zhang", [
        {"id": "step_1_grep", "kind": "tool", "tool": "grep", "instruction": "Search for author of hotfix commit", "depends_on": []}
    ]),
    BenchmarkTask("ruler_10_multihop_8k_rel", "ruler_l1", "level_1", "What is the child task ID spawned by task_root_101?", _noise(120) + "Parent task_root_101 spawned subtask task_sub_404 successfully.\n" + _noise(110), [{"name": "grep"}], "task_sub_404", [
        {"id": "step_1_grep", "kind": "tool", "tool": "grep", "instruction": "Search subtask spawned by task_root_101", "depends_on": []}
    ]),
    BenchmarkTask("ruler_11_multihop_16k", "ruler_l1", "level_1", "Identify the server node and rule name that triggered the OOM kill event.", _noise(180) + "Host node_us_east_2 triggered alert RULE_OOM_KILL.\n" + _noise(180), [{"name": "grep"}], "RULE_OOM_KILL", [
        {"id": "step_1_grep", "kind": "tool", "tool": "grep", "instruction": "Search rule name triggering OOM kill alert", "depends_on": []}
    ]),
    BenchmarkTask("ruler_12_multihop_16k_order", "ruler_l1", "level_1", "Identify the buyer company and transaction ID for order 99201.", _noise(190) + "Order 99201: buyer=Acquired By Globex, transaction=tx_pay_88190.\n" + _noise(180), [{"name": "grep"}], "Globex", [
        {"id": "step_1_grep", "kind": "tool", "tool": "grep", "instruction": "Search buyer for order 99201", "depends_on": []}
    ]),
    BenchmarkTask("ruler_13_multihop_32k", "ruler_l1", "level_1", "What was the root cause hardware failure identified in incident report #402?", _noise(260) + "Incident #402 Root Cause: disk_controller_failure on rack_c7.\n" + _noise(260), [{"name": "grep"}], "disk_controller_failure", [
        {"id": "step_1_grep", "kind": "tool", "tool": "grep", "instruction": "Search root cause of incident #402", "depends_on": []}
    ]),
    BenchmarkTask("ruler_14_aggregation_8k", "ruler_l1", "level_1", "State the total count of distinct error codes recorded in the error log.", _noise(110) + "Errors: ERR_401, ERR_404, ERR_500, ERR_502. All 4 distinct errors acknowledged.\n" + _noise(110), [{"name": "calc"}], "4", [
        {"id": "step_1_count", "kind": "transform", "instruction": "Count distinct error codes listed in log", "depends_on": []}
    ]),
    BenchmarkTask("ruler_15_aggregation_16k", "ruler_l1", "level_1", "What is the most frequent telemetry event code observed in the system stream?", _noise(180) + "Summary: EVENT_DISCONNECT: 2, EVENT_TIMEOUT: 5, HEARTBEAT_ACK: 1042.\n" + _noise(180), [{"name": "grep"}], "HEARTBEAT_ACK", [
        {"id": "step_1_grep", "kind": "tool", "tool": "grep", "instruction": "Identify most frequent telemetry event", "depends_on": []}
    ]),

    # 3. SWE Level 1 (10 tasks)
    BenchmarkTask("swe_01_division_zero", "swe_l1", "level_1", "Fix ZeroDivisionError in compute_average(items) when items list is empty, returning 0.0.", "def compute_average(items): return sum(items)/len(items) if items else 0.0", [], "0.0", [
        {"id": "step_1_fix", "kind": "transform", "instruction": "Provide safe average fallback return value", "depends_on": []}
    ]),
    BenchmarkTask("swe_02_json_schema_repair", "swe_l1", "level_1", "Identify missing required receipt digest field in schema.", "Receipt requires: run_id, status, receipt_sha256.", [{"name": "json_filter"}], "receipt_sha256", [
        {"id": "step_1_field", "kind": "transform", "instruction": "Identify required receipt digest field name", "depends_on": []}
    ]),
    BenchmarkTask("swe_03_list_index_error", "swe_l1", "level_1", "What should pop() on an empty stack return when guarded safely with default?", "def safe_pop(stack): return stack.pop() if stack else None", [], "None", [
        {"id": "step_1_pop", "kind": "transform", "instruction": "Identify safe default return value for empty pop", "depends_on": []}
    ]),
    BenchmarkTask("swe_04_sql_injection", "swe_l1", "level_1", "Which dangerous technique should be replaced by parameterized queries: string formatting or cursor binding?", "Review: query = f'SELECT * FROM users WHERE id = {user_id}'", [], "string formatting", [
        {"id": "step_1_sec", "kind": "transform", "instruction": "Identify vulnerable technique name", "depends_on": []}
    ]),
    BenchmarkTask("swe_05_regex_escape", "swe_l1", "level_1", "What character must be escaped with a backslash to match a literal period in regex?", "Regex problem: pattern = 'www.google.com'", [], "\\.", [
        {"id": "step_1_regex", "kind": "transform", "instruction": "Provide literal period regex escape sequence", "depends_on": []}
    ]),
    BenchmarkTask("swe_06_dict_get_fallback", "swe_l1", "level_1", "Which dictionary method safely retrieves a value with a fallback default?", "Method options: dict.pop, dict.keys, dict.get.", [], "dict.get", [
        {"id": "step_1_method", "kind": "transform", "instruction": "Identify safe dictionary lookup method", "depends_on": []}
    ]),
    BenchmarkTask("swe_07_type_hint_fix", "swe_l1", "level_1", "Specify the proper typing annotation for a variable that can be string or None.", "Typing options: Union[str, int], Optional[str], List[str].", [], "Optional[str]", [
        {"id": "step_1_hint", "kind": "transform", "instruction": "Select proper type hint annotation", "depends_on": []}
    ]),
    BenchmarkTask("swe_08_retry_decorator", "swe_l1", "level_1", "What parameter name sets the maximum retry attempts in the tenacity retry decorator?", "Example: @retry(stop=stop_after_attempt(3), max_retries=3)", [], "max_retries=3", [
        {"id": "step_1_retry", "kind": "transform", "instruction": "Identify parameter name setting retry limit", "depends_on": []}
    ]),
    BenchmarkTask("swe_09_datetime_iso", "swe_l1", "level_1", "What is the standard ISO8601 representation of midnight UTC on September 16, 2026?", "Format requirement: YYYY-MM-DDTHH:MM:SSZ", [], "2026-09-16T00:00:00Z", [
        {"id": "step_1_iso", "kind": "transform", "instruction": "Provide exact ISO8601 midnight timestamp", "depends_on": []}
    ]),
    BenchmarkTask("swe_10_subprocess_check", "swe_l1", "level_1", "Which parameter in subprocess.run ensures an exception is raised on non-zero exit code?", "Options: capture_output=True, check=True, shell=True.", [], "check=True", [
        {"id": "step_1_check", "kind": "transform", "instruction": "Identify subprocess argument raising on failure", "depends_on": []}
    ]),

    # =========================================================================
    # TIER 2: LEVEL 2 ADVANCED (36 tasks)
    # =========================================================================

    # 4. DAG Tool Chaining (8 tasks)
    BenchmarkTask(
        task_id="dag_01_db_decrypt_verify",
        suite="dag_chain",
        tier="level_2",
        prompt="Query user vault record for user_819, decrypt the base64 token, and output the decrypted authentication string.",
        context="Vault server: vault.pdx.internal:8200",
        tools=[{"name": "db_lookup"}, {"name": "crypto_transform"}],
        expected_answer="PDX_AUTH_9921",
        pdx_steps=[
            {"id": "step_1_lookup", "kind": "tool", "tool": "db_lookup", "instruction": "Query db_lookup for 'user_819'", "depends_on": []},
            {"id": "step_2_decrypt", "kind": "tool", "tool": "crypto_transform", "instruction": "Decode base64 payload from step_1_lookup", "depends_on": ["step_1_lookup"]},
            {"id": "step_3_verify", "kind": "verify", "instruction": "Extract authentication token string", "depends_on": ["step_2_decrypt"]},
        ],
    ),
    BenchmarkTask(
        task_id="dag_02_metrics_variance_calc",
        suite="dag_chain",
        tier="level_2",
        prompt="Lookup latency metrics for cluster 'cl_alpha', calculate the mean latency, and then calculate max deviation from the mean.",
        context="Cluster telemetry registry available via db_lookup.",
        tools=[{"name": "db_lookup"}, {"name": "calc"}],
        expected_answer="18.0",
        pdx_steps=[
            {"id": "step_1_metrics", "kind": "tool", "tool": "db_lookup", "instruction": "Lookup latency list for 'cl_alpha'", "depends_on": []},
            {"id": "step_2_mean", "kind": "tool", "tool": "calc", "instruction": "Compute average latency from step_1_metrics", "depends_on": ["step_1_metrics"]},
            {"id": "step_3_deviation", "kind": "tool", "tool": "calc", "instruction": "Compute max deviation (160 - mean)", "depends_on": ["step_2_mean"]},
        ],
    ),
    BenchmarkTask(
        task_id="dag_03_log_filter_aggregate",
        suite="dag_chain",
        tier="level_2",
        prompt="Filter server log for incidents with status >= 500, then calculate total affected users by summing affected_users across matching incidents.",
        context="Incident logs: [{'id': 101, 'status': 503, 'affected_users': 210}, {'id': 102, 'status': 200, 'affected_users': 0}, {'id': 103, 'status': 500, 'affected_users': 215}]",
        tools=[{"name": "json_filter"}, {"name": "calc"}],
        expected_answer="425",
        pdx_steps=[
            {"id": "step_1_filter", "kind": "tool", "tool": "json_filter", "instruction": "Filter incidents where status >= 500", "depends_on": []},
            {"id": "step_2_sum", "kind": "tool", "tool": "calc", "instruction": "Sum affected_users from step_1_filter: 210 + 215", "depends_on": ["step_1_filter"]},
        ],
    ),
    BenchmarkTask(
        task_id="dag_04_api_chain_exchange",
        suite="dag_chain",
        tier="level_2",
        prompt="Lookup exchange rates EUR->USD and USD->JPY from forex database, then compute the exact JPY conversion for 500 EUR.",
        context="Forex gateway online.",
        tools=[{"name": "db_lookup"}, {"name": "calc"}],
        expected_answer="83700",
        pdx_steps=[
            {"id": "step_1_forex", "kind": "tool", "tool": "db_lookup", "instruction": "Lookup forex rates EUR_USD and USD_JPY", "depends_on": []},
            {"id": "step_2_convert", "kind": "tool", "tool": "calc", "instruction": "Compute 500 * EUR_USD * USD_JPY (500 * 1.08 * 155.0)", "depends_on": ["step_1_forex"]},
        ],
    ),
    BenchmarkTask(
        task_id="dag_05_auth_hash_signature",
        suite="dag_chain",
        tier="level_2",
        prompt="Lookup secret salt for service 'auth_gate', concatenate with session token 'sess_8892', and compute sha256 digest prefix.",
        context="Auth security config partition.",
        tools=[{"name": "db_lookup"}, {"name": "crypto_transform"}],
        expected_answer="e5a190b2",
        pdx_steps=[
            {"id": "step_1_salt", "kind": "tool", "tool": "db_lookup", "instruction": "Lookup salt for auth_gate", "depends_on": []},
            {"id": "step_2_hash", "kind": "tool", "tool": "crypto_transform", "instruction": "Compute sha256 of sess_8892:SALT_772", "depends_on": ["step_1_salt"]},
        ],
    ),
    BenchmarkTask(
        task_id="dag_06_inventory_reorder_qty",
        suite="dag_chain",
        tier="level_2",
        prompt="Lookup stock of 'SKU_9021' (current=45, target=100, batch_size=20), then calculate how many integer batches must be ordered to reach or exceed target.",
        context="ERP inventory warehouse 3.",
        tools=[{"name": "db_lookup"}, {"name": "calc"}],
        expected_answer="3",
        pdx_steps=[
            {"id": "step_1_stock", "kind": "tool", "tool": "db_lookup", "instruction": "Lookup SKU_9021 parameters", "depends_on": []},
            {"id": "step_2_calc", "kind": "tool", "tool": "calc", "instruction": "Compute ceil((100 - 45) / 20)", "depends_on": ["step_1_stock"]},
        ],
    ),
    BenchmarkTask(
        task_id="dag_07_cas_blob_verify",
        suite="dag_chain",
        tier="level_2",
        prompt="Lookup CAS manifest for artifact 'pkg_linux_x64', verify checksum status, and report root verification status.",
        context="CAS Storage Cluster A.",
        tools=[{"name": "db_lookup"}, {"name": "crypto_transform"}],
        expected_answer="ROOT_MERKLE_OK",
        pdx_steps=[
            {"id": "step_1_manifest", "kind": "tool", "tool": "db_lookup", "instruction": "Lookup manifest for pkg_linux_x64", "depends_on": []},
            {"id": "step_2_verify", "kind": "verify", "instruction": "Check merkle root match", "depends_on": ["step_1_manifest"]},
        ],
    ),
    BenchmarkTask(
        task_id="dag_08_pipeline_stage_dependency",
        suite="dag_chain",
        tier="level_2",
        prompt="Trace job pipeline for 'deploy_prod', find which dependent stage failed and prevented deployment.",
        context="CI Pipeline trace: build -> unit_test -> lint_check (FAILED) -> integration_test (SKIPPED) -> deploy_prod (BLOCKED)",
        tools=[{"name": "json_filter"}],
        expected_answer="lint_check",
        pdx_steps=[
            {"id": "step_1_trace", "kind": "tool", "tool": "json_filter", "instruction": "Filter failed pipeline stages", "depends_on": []},
            {"id": "step_2_resolve", "kind": "verify", "instruction": "Identify blocking failed stage name", "depends_on": ["step_1_trace"]},
        ],
    ),

    # 5. GAIA Level 2 Multi-Tool Challenges (8 tasks)
    BenchmarkTask(
        task_id="gaia_l2_01_multisheet_finance",
        suite="gaia_l2",
        tier="level_2",
        prompt="From financial statement, compute Net Profit Margin: Total Revenue is $450M, Total Operating Costs are $310M, Tax is $57.2M. Margin = (Revenue - Costs - Tax) / Revenue.",
        context="Annual Statement: Revenue=$450,000,000; Operating_Cost=$310,000,000; Tax=$57,200,000.",
        tools=[{"name": "calc"}],
        expected_answer="18.4%",
        pdx_steps=[
            {"id": "step_1_net_profit", "kind": "tool", "tool": "calc", "instruction": "Compute 450 - 310 - 57.2", "depends_on": []},
            {"id": "step_2_margin", "kind": "tool", "tool": "calc", "instruction": "Compute (82.8 / 450) * 100", "depends_on": ["step_1_net_profit"]},
        ],
    ),
    BenchmarkTask(
        task_id="gaia_l2_02_server_fleet_capacity",
        suite="gaia_l2",
        tier="level_2",
        prompt="Determine minimum hourly cost to provision at least 128 vCPUs and 512GB RAM using fleet instances: type_a (32 vCPU, 128GB, $3.60/hr) vs type_b (64 vCPU, 256GB, $7.10/hr).",
        context="Cloud catalog: type_a=$3.60/hr (32 vCPU, 128GB RAM), type_b=$7.10/hr (64 vCPU, 256GB RAM). Need 128 vCPU, 512GB RAM.",
        tools=[{"name": "calc"}],
        expected_answer="$14.20",
        pdx_steps=[
            {"id": "step_1_cost_a", "kind": "tool", "tool": "calc", "instruction": "Compute 4 * 3.60", "depends_on": []},
            {"id": "step_2_cost_b", "kind": "tool", "tool": "calc", "instruction": "Compute 2 * 7.10", "depends_on": []},
            {"id": "step_3_min", "kind": "verify", "instruction": "Select minimal cost", "depends_on": ["step_1_cost_a", "step_2_cost_b"]},
        ],
    ),
    BenchmarkTask(
        task_id="gaia_l2_03_time_zone_flight",
        suite="gaia_l2",
        tier="level_2",
        prompt="A flight departs Tokyo (UTC+9) at 14:30 on Oct 1. Flight time is 9 hours 45 minutes. Arrival city is San Francisco (UTC-7). What is local arrival time in San Francisco?",
        context="Tokyo UTC+9, San Francisco UTC-7. Time difference = 16 hours behind Tokyo.",
        tools=[{"name": "calc"}],
        expected_answer="08:15",
        pdx_steps=[
            {"id": "step_1_utc", "kind": "tool", "tool": "calc", "instruction": "Calculate departure in UTC: 14.5 - 9 = 5.5 (05:30 UTC)", "depends_on": []},
            {"id": "step_2_arrival_utc", "kind": "tool", "tool": "calc", "instruction": "Add flight duration: 05:30 + 9h45m = 15:15 UTC", "depends_on": ["step_1_utc"]},
            {"id": "step_3_local", "kind": "tool", "tool": "calc", "instruction": "Convert to UTC-7: 15:15 - 7 hours = 08:15", "depends_on": ["step_2_arrival_utc"]},
        ],
    ),
    BenchmarkTask(
        task_id="gaia_l2_04_packet_loss_sla",
        suite="gaia_l2",
        tier="level_2",
        prompt="Examine 5 hourly network reports: [0.1%, 0.8%, 0.4%, 1.2%, 0.9%]. If threshold is >0.5%, how many hours breached, and is SLA violated (violated if breaches >= 3)?",
        context="SLA Contract clause 4.2: Max allowed breach hours is 2.",
        tools=[{"name": "calc"}],
        expected_answer="3 hours, breached",
        pdx_steps=[
            {"id": "step_1_count", "kind": "tool", "tool": "calc", "instruction": "Count elements > 0.5 in [0.1, 0.8, 0.4, 1.2, 0.9]", "depends_on": []},
            {"id": "step_2_sla", "kind": "verify", "instruction": "Check if count >= 3", "depends_on": ["step_1_count"]},
        ],
    ),
    BenchmarkTask(
        task_id="gaia_l2_05_energy_consumption_bill",
        suite="gaia_l2",
        tier="level_2",
        prompt="Calculate total electricity bill for 450 kWh with tiered rates: Tier 1 (first 100 kWh @ $0.12), Tier 2 (next 200 kWh @ $0.18), Tier 3 (remaining 150 kWh @ $0.25).",
        context="Tariff schedule 2026.",
        tools=[{"name": "calc"}],
        expected_answer="$85.50",
        pdx_steps=[
            {"id": "step_1_tier_calc", "kind": "tool", "tool": "calc", "instruction": "Compute 100*0.12 + 200*0.18 + 150*0.25", "depends_on": []},
        ],
    ),
    BenchmarkTask(
        task_id="gaia_l2_06_docker_layer_cache",
        suite="gaia_l2",
        tier="level_2",
        prompt="Identify the exact Dockerfile instruction that invalidated layer cache in build log.",
        context="Build log:\nStep 1/5: FROM python:3.12 (CACHED)\nStep 2/5: WORKDIR /app (CACHED)\nStep 3/5: COPY requirements.txt . (CACHED)\nStep 4/5: RUN pip install -r requirements.txt (CACHE MISS)\nStep 5/5: COPY . .",
        tools=[{"name": "grep"}],
        expected_answer="RUN pip install -r requirements.txt",
        pdx_steps=[
            {"id": "step_1_find_miss", "kind": "tool", "tool": "grep", "instruction": "Find line with CACHE MISS", "depends_on": []},
        ],
    ),
    BenchmarkTask(
        task_id="gaia_l2_07_geo_distance_shipping",
        suite="gaia_l2",
        tier="level_2",
        prompt="Given warehouse at X=0, Y=0 and recipient at X=120km, Y=160km, calculate Euclidean distance and assign shipping zone (<100km: Zone 1, 100-250km: Zone 2, >250km: Zone 3).",
        context="Logistics coordinate grid.",
        tools=[{"name": "calc"}],
        expected_answer="Zone 2",
        pdx_steps=[
            {"id": "step_1_distance", "kind": "tool", "tool": "calc", "instruction": "Compute sqrt(120**2 + 160**2) = 200km", "depends_on": []},
            {"id": "step_2_zone", "kind": "verify", "instruction": "Classify 200km into Zone", "depends_on": ["step_1_distance"]},
        ],
    ),
    BenchmarkTask(
        task_id="gaia_l2_08_semver_compatibility",
        suite="gaia_l2",
        tier="level_2",
        prompt="Given semver constraint '^2.4.0', which is the highest compatible release among ['2.3.9', '2.4.1', '2.4.8', '2.5.0', '3.0.0']?",
        context="Semver carets allow minor/patch bumps within major version 2.",
        tools=[{"name": "json_filter"}],
        expected_answer="2.5.0",
        pdx_steps=[
            {"id": "step_1_filter", "kind": "tool", "tool": "json_filter", "instruction": "Filter versions matching ^2.4.0 (>=2.4.0 <3.0.0)", "depends_on": []},
            {"id": "step_2_max", "kind": "verify", "instruction": "Pick max version: 2.5.0", "depends_on": ["step_1_filter"]},
        ],
    ),

    # 6. SWE Algorithmic Repair with Hidden Unit Tests (8 tasks)
    BenchmarkTask(
        task_id="swe_algo_01_binary_search_bounds",
        suite="swe_algo",
        tier="level_2",
        prompt="Fix the off-by-one error in binary_search(arr, target) when updating low pointer upon target > arr[mid]. What should low be set to?",
        context="Buggy code:\nif arr[mid] < target:\n    low = mid  # BUG: causes infinite loop on small arrays\nelse:\n    high = mid - 1",
        tools=[],
        expected_answer="low = mid + 1",
        pdx_steps=[
            {"id": "step_1_fix", "kind": "transform", "instruction": "Identify standard binary search lower bound update statement", "depends_on": []},
        ],
    ),
    BenchmarkTask(
        task_id="swe_algo_02_lru_cache_eviction",
        suite="swe_algo",
        tier="level_2",
        prompt="In an OrderedDict-based LRU cache, which method call is required when a key is hit to mark it as most recently used?",
        context="class LRUCache:\n    def get(self, key):\n        if key not in self.cache: return -1\n        # Missing line to mark recent\n        return self.cache[key]",
        tools=[],
        expected_answer="move_to_end",
        pdx_steps=[
            {"id": "step_1_identify", "kind": "transform", "instruction": "Identify OrderedDict method to update item recency", "depends_on": []},
        ],
    ),
    BenchmarkTask(
        task_id="swe_algo_03_cycle_detection_floyd",
        suite="swe_algo",
        tier="level_2",
        prompt="In Floyd's cycle detection algorithm, what condition must guard 'fast = fast.next.next' in the while loop condition to prevent AttributeError?",
        context="while fast and fast.next:\n    slow = slow.next\n    fast = fast.next.next",
        tools=[],
        expected_answer="fast and fast.next",
        pdx_steps=[
            {"id": "step_1_guard", "kind": "transform", "instruction": "State the pointer guard condition", "depends_on": []},
        ],
    ),
    BenchmarkTask(
        task_id="swe_algo_04_merge_intervals",
        suite="swe_algo",
        tier="level_2",
        prompt="Before iterating to merge overlapping intervals [[x1, y1], [x2, y2]], what sorting statement is mandatory?",
        context="def merge_intervals(intervals):\n    # Missing sort statement!\n    merged = []\n    for intv in intervals: ...",
        tools=[],
        expected_answer="intervals.sort(key=lambda x: x[0])",
        pdx_steps=[
            {"id": "step_1_sort", "kind": "transform", "instruction": "Provide python interval sorting statement", "depends_on": []},
        ],
    ),
    BenchmarkTask(
        task_id="swe_algo_05_json_cas_hash_deterministic",
        suite="swe_algo",
        tier="level_2",
        prompt="To ensure json.dumps() produces a deterministic SHA-256 digest regardless of key insertion order, which argument must be set to True?",
        context="import json, hashlib\ndef digest(d): return hashlib.sha256(json.dumps(d, ???).encode()).hexdigest()",
        tools=[],
        expected_answer="sort_keys=True",
        pdx_steps=[
            {"id": "step_1_flag", "kind": "transform", "instruction": "Identify deterministic json key ordering argument", "depends_on": []},
        ],
    ),
    BenchmarkTask(
        task_id="swe_algo_06_tree_max_depth",
        suite="swe_algo",
        tier="level_2",
        prompt="What is the standard base case return for max_depth(root) of a binary tree when root is None?",
        context="def max_depth(root):\n    if not root:\n        return ???",
        tools=[],
        expected_answer="0",
        pdx_steps=[
            {"id": "step_1_base", "kind": "transform", "instruction": "Identify base case value for empty tree depth", "depends_on": []},
        ],
    ),
    BenchmarkTask(
        task_id="swe_algo_07_rate_limiter_token_bucket",
        suite="swe_algo",
        tier="level_2",
        prompt="In a token bucket rate limiter, what python function prevents replenished tokens from exceeding max bucket capacity?",
        context="self.tokens = ???(self.capacity, self.tokens + added_tokens)",
        tools=[],
        expected_answer="min",
        pdx_steps=[
            {"id": "step_1_func", "kind": "transform", "instruction": "Identify boundary clamp function", "depends_on": []},
        ],
    ),
    BenchmarkTask(
        task_id="swe_algo_08_topological_sort_kahn",
        suite="swe_algo",
        tier="level_2",
        prompt="In Kahn's algorithm for topological sort, when a node's in-degree reaches what integer value is it appended to the zero-in-degree queue?",
        context="for neighbor in graph[node]:\n    in_degree[neighbor] -= 1\n    if in_degree[neighbor] == ???:\n        queue.append(neighbor)",
        tools=[{"name": "run_python"}],
        expected_answer="0",
        pdx_steps=[
            {"id": "step_1_check", "kind": "transform", "instruction": "Identify queue push condition for Kahn's algorithm", "depends_on": []},
        ],
    ),

    # 7. GSM8K-Hard Multi-Step Arithmetic (6 tasks)
    BenchmarkTask(
        task_id="gsm_01_compound_depreciation",
        suite="gsm8k_hard",
        tier="level_2",
        prompt="A server cluster originally purchased for $12,000 depreciates by 15% each year. What is its exact residual book value at the end of 3 years?",
        context="Depreciation formula: Value = Principal * (1 - rate)^years = 12000 * (0.85)^3.",
        tools=[{"name": "calc"}],
        expected_answer="7369.5",
        pdx_steps=[
            {"id": "step_1_deprec", "kind": "tool", "tool": "calc", "instruction": "Compute 12000 * (0.85 ** 3)", "depends_on": []},
        ],
    ),
    BenchmarkTask(
        task_id="gsm_02_compound_interest_quarterly",
        suite="gsm8k_hard",
        tier="level_2",
        prompt="A $5,000 investment earns 8% annual interest compounded quarterly for 2 years. Compute the final amount using A = P * (1 + r/n)^(n*t).",
        context="P=5000, r=0.08, n=4, t=2 -> A = 5000 * (1 + 0.02)^8.",
        tools=[{"name": "calc"}],
        expected_answer="5858.3",
        pdx_steps=[
            {"id": "step_1_interest", "kind": "tool", "tool": "calc", "instruction": "Compute round(5000 * (1.02 ** 8), 1)", "depends_on": []},
        ],
    ),
    BenchmarkTask(
        task_id="gsm_03_worker_productivity_shifts",
        suite="gsm8k_hard",
        tier="level_2",
        prompt="12 workers produce 360 units in 5 days of 6-hour shifts. How many total units will 8 workers produce in 9 days of 8-hour shifts at the same productivity rate?",
        context="Rate = 360 / (12 * 5 * 6) = 1.0 unit / worker-hour. New = 8 * 9 * 8 * 1.0.",
        tools=[{"name": "calc"}],
        expected_answer="576",
        pdx_steps=[
            {"id": "step_1_rate", "kind": "tool", "tool": "calc", "instruction": "Compute 360 / (12 * 5 * 6)", "depends_on": []},
            {"id": "step_2_total", "kind": "tool", "tool": "calc", "instruction": "Compute 8 * 9 * 8 * 1.0", "depends_on": ["step_1_rate"]},
        ],
    ),
    BenchmarkTask(
        task_id="gsm_04_multitier_commission",
        suite="gsm8k_hard",
        tier="level_2",
        prompt="A sales rep earns 4% on first $50,000, 7% on the next $50,000, and 10% on anything exceeding $100,000. Calculate total commission for $145,000 total sales.",
        context="Tier 1: 50000*0.04; Tier 2: 50000*0.07; Tier 3: 45000*0.10.",
        tools=[{"name": "calc"}],
        expected_answer="10000",
        pdx_steps=[
            {"id": "step_1_tiers", "kind": "tool", "tool": "calc", "instruction": "Compute 50000*0.04 + 50000*0.07 + 45000*0.10", "depends_on": []},
        ],
    ),
    BenchmarkTask(
        task_id="gsm_05_mixture_concentration",
        suite="gsm8k_hard",
        tier="level_2",
        prompt="40 liters of 20% acid solution is blended with 60 liters of 35% acid solution. What is the resulting acid concentration percentage?",
        context="Pure acid = 40*0.20 + 60*0.35 = 8 + 21 = 29 liters. Total volume = 100 liters.",
        tools=[{"name": "calc"}],
        expected_answer="29%",
        pdx_steps=[
            {"id": "step_1_blend", "kind": "tool", "tool": "calc", "instruction": "Compute ((40*0.20 + 60*0.35) / 100) * 100", "depends_on": []},
        ],
    ),
    BenchmarkTask(
        task_id="gsm_06_train_relative_speed",
        suite="gsm8k_hard",
        tier="level_2",
        prompt="Train A (250m long, speed 72 km/h = 20 m/s) and Train B (350m long, speed 108 km/h = 30 m/s) travel towards each other on parallel tracks. How many seconds elapse from initial engine encounter until tails completely clear?",
        context="Total clearance length = 250 + 350 = 600m. Relative speed = 20 + 30 = 50 m/s. Time = 600 / 50.",
        tools=[{"name": "calc"}],
        expected_answer="12",
        pdx_steps=[
            {"id": "step_1_time", "kind": "tool", "tool": "calc", "instruction": "Compute (250 + 350) / (20 + 30)", "depends_on": []},
        ],
    ),

    # 8. RULER Multi-Hop Long Context (6 tasks)
    BenchmarkTask(
        task_id="ruler_01_hop2_8k_crypto",
        suite="ruler_multihop",
        tier="level_2",
        prompt="Find the active vault session identifier, then find the specific decryption key assigned to that active session.",
        context=_noise(110) + "Active session established: session_id is SESSION_OMEGA_882.\n" + _noise(80) + "Session SESSION_OMEGA_882 decrypts with key KEY_CIPHER_99014.\n" + _noise(110),
        tools=[{"name": "grep"}],
        expected_answer="KEY_CIPHER_99014",
        pdx_steps=[
            {"id": "step_1_session", "kind": "tool", "tool": "grep", "instruction": "Search for active session_id in context", "depends_on": []},
            {"id": "step_2_key", "kind": "tool", "tool": "grep", "instruction": "Search for decryption key matching session_id", "depends_on": ["step_1_session"]},
        ],
    ),
    BenchmarkTask(
        task_id="ruler_02_hop2_8k_ip_port",
        suite="ruler_multihop",
        tier="level_2",
        prompt="Find which cluster node was elected as primary replica, then identify the internal IP:port address for that primary node.",
        context=_noise(120) + "Elected primary database replica is node-mel-srv7.\n" + _noise(70) + "Host node-mel-srv7 binds to internal socket 10.144.92.11:9042.\n" + _noise(110),
        tools=[{"name": "grep"}],
        expected_answer="10.144.92.11:9042",
        pdx_steps=[
            {"id": "step_1_replica", "kind": "tool", "tool": "grep", "instruction": "Search for elected primary node", "depends_on": []},
            {"id": "step_2_socket", "kind": "tool", "tool": "grep", "instruction": "Search for socket binding of elected primary node", "depends_on": ["step_1_replica"]},
        ],
    ),
    BenchmarkTask(
        task_id="ruler_03_hop2_8k_commit_author",
        suite="ruler_multihop",
        tier="level_2",
        prompt="Find the pull request number merged for release v3.2.0, then find the name of the author of that pull request.",
        context=_noise(100) + "Release v3.2.0 incorporates security PR #1492.\n" + _noise(90) + "PR #1492 author confirmed as Sarah Chen.\n" + _noise(110),
        tools=[{"name": "grep"}],
        expected_answer="Sarah Chen",
        pdx_steps=[
            {"id": "step_1_pr", "kind": "tool", "tool": "grep", "instruction": "Search for PR number in release v3.2.0", "depends_on": []},
            {"id": "step_2_author", "kind": "tool", "tool": "grep", "instruction": "Search for author of PR", "depends_on": ["step_1_pr"]},
        ],
    ),
    BenchmarkTask(
        task_id="ruler_04_hop2_12k_secret_hash",
        suite="ruler_multihop",
        tier="level_2",
        prompt="Find the reference hash for critical incident #804, then identify the root cause hardware failure tag linked to that hash.",
        context=_noise(160) + "Critical incident #804 archived under REF_HASH_EE492.\n" + _noise(140) + "REF_HASH_EE492 root cause identified as FLASH_SSD_SLOT3_FAILED.\n" + _noise(160),
        tools=[{"name": "grep"}],
        expected_answer="FLASH_SSD_SLOT3_FAILED",
        pdx_steps=[
            {"id": "step_1_ref", "kind": "tool", "tool": "grep", "instruction": "Search for incident #804 reference hash", "depends_on": []},
            {"id": "step_2_tag", "kind": "tool", "tool": "grep", "instruction": "Search for root cause failure tag linked to reference hash", "depends_on": ["step_1_ref"]},
        ],
    ),
    BenchmarkTask(
        task_id="ruler_05_hop3_8k_token_chain",
        suite="ruler_multihop",
        tier="level_2",
        prompt="Find the storage bucket for Project Apollo, find the KMS key alias for that bucket, and output the final KMS Key ARN.",
        context=_noise(110) + "Project Apollo storage bucket is config-apollo-prd.\n" + _noise(60) + "Bucket config-apollo-prd encrypted via alias/apollo-seal-kms.\n" + _noise(70) + "alias/apollo-seal-kms resolves to ARN arn:aws:kms:us-east-1:123456789:key/9918a.\n" + _noise(100),
        tools=[{"name": "grep"}],
        expected_answer="arn:aws:kms:us-east-1:123456789:key/9918a",
        pdx_steps=[
            {"id": "step_1_bucket", "kind": "tool", "tool": "grep", "instruction": "Search bucket for Project Apollo", "depends_on": []},
            {"id": "step_2_alias", "kind": "tool", "tool": "grep", "instruction": "Search KMS alias for bucket", "depends_on": ["step_1_bucket"]},
            {"id": "step_3_arn", "kind": "tool", "tool": "grep", "instruction": "Search KMS ARN for alias", "depends_on": ["step_2_alias"]},
        ],
    ),
    BenchmarkTask(
        task_id="ruler_06_hop2_12k_agent_id",
        suite="ruler_multihop",
        tier="level_2",
        prompt="Find which worker node was assigned task TASK_DISPATCH_774, then locate the completion receipt digest recorded by that worker.",
        context=_noise(150) + "TASK_DISPATCH_774 assigned to worker Worker_Bravo_9.\n" + _noise(130) + "Worker_Bravo_9 job completion receipt digest: sha256_e4c88219b.\n" + _noise(160),
        tools=[{"name": "grep"}],
        expected_answer="sha256_e4c88219b",
        pdx_steps=[
            {"id": "step_1_worker", "kind": "tool", "tool": "grep", "instruction": "Search worker assigned to TASK_DISPATCH_774", "depends_on": []},
            {"id": "step_2_digest", "kind": "tool", "tool": "grep", "instruction": "Search completion receipt digest for worker", "depends_on": ["step_1_worker"]},
        ],
    ),
]


def load_benchmark_suite(tier: Optional[str] = None, suite_name: Optional[str] = None) -> List[BenchmarkTask]:
    tasks = BENCHMARK_TASKS
    if tier and tier != "all":
        tasks = [t for t in tasks if t.tier == tier]
    if suite_name and suite_name != "all":
        tasks = [t for t in tasks if t.suite == suite_name]
    return tasks


if __name__ == "__main__":
    tasks = load_benchmark_suite()
    l1 = [t for t in tasks if t.tier == "level_1"]
    l2 = [t for t in tasks if t.tier == "level_2"]
    print(f"Loaded {len(tasks)} benchmark tasks across 2 tiers:")
    print(f"  - Level 1 Baseline: {len(l1)} tasks (Suites: {set(t.suite for t in l1)})")
    print(f"  - Level 2 Advanced: {len(l2)} tasks (Suites: {set(t.suite for t in l2)})")


In [ ]:
# === 6. Condition Runners ===
# Condition A: GoldPlanReActRunner
"""Condition A: Gold Plan ReAct Runner (Accumulating Conversation Loop).

Strictly follows Section 2A and Section 3 of FAIR_COMPARISON_PLAN.md:
- One ongoing conversation that accumulates history over steps.
- Sees the gold execution plan and full context.
- Requires JSON schema for all turns.
- On invalid output, appends retry observation and continues.
- Never calls tools on empty or malformed outputs.
"""

from __future__ import annotations

import json
import re
import time
from typing import Any, Callable, Dict, List, Optional, Tuple


def parse_model_json(response: str, available_tools: List[str]) -> Tuple[Optional[Dict[str, Any]], str]:
    """Validate model output against Section 3 schema.
    
    Returns:
        (parsed_dict_or_None, reason_string)
    """
    if not response or not response.strip():
        return None, "empty_generation"
    
    text = response.strip()
    match = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text)
    raw_json = match.group(1).strip() if match else text
    
    start = raw_json.find("{")
    end = raw_json.rfind("}")
    if start != -1 and end != -1 and end > start:
        raw_json = raw_json[start:end+1]
    
    try:
        data = json.loads(raw_json)
    except Exception as exc:
        return None, f"json_parse_error: {exc}"
    
    if not isinstance(data, dict):
        return None, "json_not_a_dict"
    
    action = data.get("action")
    if action not in ["execute_tool", "final_answer"]:
        return None, f"invalid_action: '{action}'"
    
    if action == "execute_tool":
        tool_call = data.get("tool_call")
        if not isinstance(tool_call, dict):
            return None, "missing_or_invalid_tool_call"
        name = tool_call.get("name")
        if name not in available_tools:
            return None, f"tool_not_available: '{name}'"
        args = tool_call.get("args")
        if not isinstance(args, dict):
            return None, "tool_args_not_a_dict"
        has_val = any(v is not None and str(v).strip() != "" for v in args.values())
        if not has_val:
            return None, "empty_tool_args"
        return data, "valid"
    
    elif action == "final_answer":
        ans = data.get("answer")
        if ans is None or not str(ans).strip():
            return None, "empty_final_answer"
        return data, "valid"
    
    return None, "unknown_schema_violation"


class GoldPlanReActRunner:
    def __init__(self, generate_chat_fn: Callable[[List[Dict[str, str]], int], str]):
        self.generate_chat_fn = generate_chat_fn

    def run_task(
        self,
        task: Any,
        execute_tool_fn: Callable[[str, Dict[str, Any], str], str],
        model_name: str = "unknown_model",
        max_steps: Optional[int] = None,
    ) -> Dict[str, Any]:
        start_time = time.time()
        gold_steps = task.get_full_gold_steps()
        available_tools = [t["name"] for t in task.tools]
        step_limit = max_steps or (len(gold_steps) + 2)

        gold_plan_text = json.dumps(
            [{"id": s["id"], "instruction": s["instruction"], "depends_on": s.get("depends_on", [])} for s in gold_steps],
            indent=2
        )
        tools_desc = json.dumps(task.tools, indent=2)

        system_prompt = (
            "You are an assistant executing a task according to a gold plan.\n"
            "You must respond ONLY with valid JSON conforming to the following schema:\n"
            "{\n"
            '  "thought": "one short sentence",\n'
            '  "action": "execute_tool" | "final_answer",\n'
            '  "tool_call": {"name": "<tool>", "args": {"key": "val"}},\n'
            '  "answer": "<string>"\n'
            "}\n\n"
            f"Available Tools:\n{tools_desc}\n\n"
            f"Gold Plan:\n{gold_plan_text}\n\n"
            "Rules:\n"
            "1. Output ONLY the JSON object. No Markdown conversation, no preamble.\n"
            "2. If action is 'execute_tool', 'tool_call' must specify a valid tool and non-empty args.\n"
            "3. If action is 'final_answer', 'answer' must contain the final answer string.\n"
        )

        user_prompt = (
            f"Goal: {task.prompt}\n\n"
            f"Context:\n{task.context}\n\n"
            "Execute the task steps and provide the final answer."
        )

        messages: List[Dict[str, str]] = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]

        step_count = 0
        invalid_outputs = 0
        empty_generations = 0
        final_answer = ""
        context_growth = []

        while step_count < step_limit:
            step_count += 1
            total_chars = sum(len(m["content"]) for m in messages)
            context_growth.append(total_chars)

            raw_response = self.generate_chat_fn(messages, 256)
            messages.append({"role": "assistant", "content": raw_response})

            parsed, reason = parse_model_json(raw_response, available_tools)

            if parsed is None:
                invalid_outputs += 1
                if reason == "empty_generation":
                    empty_generations += 1
                messages.append({
                    "role": "user",
                    "content": f"Observation: invalid_output ({reason}); emit valid JSON schema with action and non-empty args."
                })
                continue

            action = parsed["action"]
            if action == "final_answer":
                final_answer = str(parsed["answer"]).strip()
                break
            elif action == "execute_tool":
                tool_call = parsed["tool_call"]
                obs = execute_tool_fn(tool_call["name"], tool_call["args"], task.context)
                messages.append({"role": "user", "content": f"Observation: {obs}"})

        elapsed_s = round(time.time() - start_time, 3)
        context_chars_last = sum(len(m["content"]) for m in messages)

        return {
            "model": model_name,
            "mode": "goldplan_react",
            "task_id": task.task_id,
            "suite": task.suite,
            "tier": task.tier,
            "success": False,  # Will be evaluated by scorer
            "invalid_outputs": invalid_outputs,
            "empty_generations": empty_generations,
            "steps_taken": step_count,
            "elapsed_s": elapsed_s,
            "final_answer": final_answer,
            "expected_answer": task.expected_answer,
            "context_chars_last": context_chars_last,
            "context_growth": context_growth,
        }


# Condition B: GoldPlanIsolatedRunner
"""Condition B: Gold Plan Isolated Dispatch Runner (Fresh O(1) Context Per Step).

Strictly follows Section 2B and Section 3 of FAIR_COMPARISON_PLAN.md:
- One fresh messages list per gold step.
- Context receives only: goal, current step instruction, parent receipts (artifact + sha256), and full context.
- No earlier assistant turns are carried forward.
- Same JSON output schema, same tools, same terminal answer requirement.
- Never calls tools on empty or malformed outputs.
"""

from __future__ import annotations

import hashlib
import json
import time
from typing import Any, Callable, Dict, List, Optional

# parse_model_json already defined above


class GoldPlanIsolatedRunner:
    def __init__(self, generate_chat_fn: Callable[[List[Dict[str, str]], int], str]):
        self.generate_chat_fn = generate_chat_fn

    def run_task(
        self,
        task: Any,
        execute_tool_fn: Callable[[str, Dict[str, Any], str], str],
        model_name: str = "unknown_model",
        max_steps: Optional[int] = None,
    ) -> Dict[str, Any]:
        start_time = time.time()
        gold_steps = task.get_full_gold_steps()
        available_tools = [t["name"] for t in task.tools]
        tools_desc = json.dumps(task.tools, indent=2)

        receipts: Dict[str, Dict[str, Any]] = {}
        step_count = 0
        invalid_outputs = 0
        empty_generations = 0
        final_answer = ""
        context_growth = []

        system_prompt = (
            "You are an isolated step execution agent.\n"
            "You must respond ONLY with valid JSON conforming to the following schema:\n"
            "{\n"
            '  "thought": "one short sentence",\n'
            '  "action": "execute_tool" | "final_answer",\n'
            '  "tool_call": {"name": "<tool>", "args": {"key": "val"}},\n'
            '  "answer": "<string>"\n'
            "}\n\n"
            f"Available Tools:\n{tools_desc}\n\n"
            "Rules:\n"
            "1. Output ONLY the JSON object. No Markdown conversation, no preamble.\n"
            "2. If action is 'execute_tool', 'tool_call' must specify a valid tool and non-empty args.\n"
            "3. If action is 'final_answer', 'answer' must contain the final answer string.\n"
        )

        for step in gold_steps:
            step_count += 1
            step_id = step["id"]
            instruction = step.get("instruction", "")
            depends_on = step.get("depends_on", [])

            # Format parent receipts
            parent_receipts_text = ""
            if depends_on:
                receipt_lines = []
                for pid in depends_on:
                    r = receipts.get(pid, {"artifact": "", "sha256": "none", "status": "missing"})
                    receipt_lines.append(f"- Step '{pid}': artifact='{r['artifact']}' (sha256={r['sha256']}, status={r['status']})")
                parent_receipts_text = "Parent Step Receipts:\n" + "\n".join(receipt_lines) + "\n\n"

            user_prompt = (
                f"Goal: {task.prompt}\n\n"
                f"Context:\n{task.context}\n\n"
                f"{parent_receipts_text}"
                f"Current Step: {step_id}\n"
                f"Step Instruction: {instruction}\n\n"
                "Execute this step and return your JSON decision."
            )

            # Fresh messages for this step!
            step_messages: List[Dict[str, str]] = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ]

            step_chars = sum(len(m["content"]) for m in step_messages)
            context_growth.append(step_chars)

            raw_response = self.generate_chat_fn(step_messages, 256)
            parsed, reason = parse_model_json(raw_response, available_tools)

            if parsed is None:
                invalid_outputs += 1
                if reason == "empty_generation":
                    empty_generations += 1
                receipts[step_id] = {
                    "artifact": "",
                    "sha256": "invalid",
                    "status": "invalid_output",
                    "reason": reason,
                }
                continue

            action = parsed["action"]
            if action == "final_answer":
                ans = str(parsed["answer"]).strip()
                final_answer = ans
                receipts[step_id] = {
                    "artifact": ans,
                    "sha256": hashlib.sha256(ans.encode("utf-8")).hexdigest()[:16],
                    "status": "completed",
                }
            elif action == "execute_tool":
                tool_call = parsed["tool_call"]
                obs = execute_tool_fn(tool_call["name"], tool_call["args"], task.context)
                digest = hashlib.sha256(str(obs).encode("utf-8")).hexdigest()[:16]
                receipts[step_id] = {
                    "artifact": str(obs),
                    "sha256": digest,
                    "status": "completed",
                }

        elapsed_s = round(time.time() - start_time, 3)
        context_chars_last = context_growth[-1] if context_growth else 0

        return {
            "model": model_name,
            "mode": "goldplan_isolated",
            "task_id": task.task_id,
            "suite": task.suite,
            "tier": task.tier,
            "success": False,  # Will be evaluated by scorer
            "invalid_outputs": invalid_outputs,
            "empty_generations": empty_generations,
            "steps_taken": step_count,
            "elapsed_s": elapsed_s,
            "final_answer": final_answer,
            "expected_answer": task.expected_answer,
            "context_chars_last": context_chars_last,
            "context_growth": context_growth,
        }


In [ ]:
# === 7. Candidate Models & Execution Loop ===
import time
import os
import gc

CANDIDATE_MODELS = [
    {"name": "qwen2.5_coder_1.5b", "model_id": "Qwen/Qwen2.5-Coder-1.5B-Instruct"},
    {"name": "qwen2.5_coder_3b", "model_id": "Qwen/Qwen2.5-Coder-3B-Instruct"},
    {"name": "llama3.2_1b", "model_id": "unsloth/Llama-3.2-1B-Instruct"},
    {"name": "llama3.2_3b", "model_id": "unsloth/Llama-3.2-3B-Instruct"},
    {"name": "gemma3_1b", "model_id": "unsloth/gemma-3-1b-it"},
    {"name": "gemma3_4b", "model_id": "unsloth/gemma-3-4b-it", "smoke_test_required": True}
]

ledger_path = "/kaggle/working/benchmark_ledger.jsonl"
os.makedirs("/kaggle/working", exist_ok=True)
if not os.path.exists(ledger_path):
    with open(ledger_path, "w", encoding="utf-8") as f:
        pass

all_eval_results = []
model_statuses = {}

for cand in CANDIDATE_MODELS:
    model_name = cand["name"]
    model_id = cand["model_id"]
    smoke_required = cand.get("smoke_test_required", False)

    print(f"\n=================== Loading Model: {model_name} ({model_id}) ===================")
    gc.collect()
    torch.cuda.empty_cache()

    model = None
    tokenizer = None
    processor = None
    is_processor = False

    try:
        if "gemma-3-4b" in model_id.lower():
            try:
                from transformers import AutoProcessor, Gemma3ForConditionalGeneration
                processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
                model = Gemma3ForConditionalGeneration.from_pretrained(
                    model_id,
                    torch_dtype=torch.float16,
                    device_map="auto",
                    trust_remote_code=True
                )
                is_processor = True
                print(f"Loaded {model_name} via Gemma3ForConditionalGeneration.")
            except Exception as ex_gem:
                print(f"Gemma 3 processor load fallback ({ex_gem}); using AutoModelForCausalLM...")
                from transformers import AutoTokenizer, AutoModelForCausalLM
                tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
                if tokenizer.pad_token is None:
                    tokenizer.pad_token = tokenizer.eos_token
                model = AutoModelForCausalLM.from_pretrained(
                    model_id,
                    torch_dtype=torch.float16,
                    device_map="auto",
                    trust_remote_code=True
                )
                is_processor = False
        else:
            from transformers import AutoTokenizer, AutoModelForCausalLM
            tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                torch_dtype=torch.float16,
                device_map="auto",
                trust_remote_code=True
            )
            is_processor = False

        model.eval()
        print(f"Model {model_name} successfully loaded.")
    except Exception as e:
        print(f"CRITICAL: Failed to load {model_name}: {e}")
        model_statuses[model_name] = f"load_failed: {e}"
        continue

    def generate_chat_fn(messages, max_tokens=256) -> str:
        if model is None:
            return ""
        try:
            if is_processor:
                proc_messages = []
                for m in messages:
                    c = m["content"]
                    if isinstance(c, str):
                        proc_messages.append({"role": m["role"], "content": [{"type": "text", "text": c}]})
                    else:
                        proc_messages.append(m)
                inputs = processor.apply_chat_template(
                    proc_messages,
                    add_generation_prompt=True,
                    tokenize=True,
                    return_dict=True,
                    return_tensors="pt"
                ).to(model.device)
                with torch.no_grad():
                    output_tokens = model.generate(
                        **inputs,
                        max_new_tokens=max_tokens,
                        do_sample=False
                    )
                input_len = inputs["input_ids"].shape[1]
                if output_tokens.shape[1] > input_len:
                    gen_ids = output_tokens[0, input_len:]
                else:
                    gen_ids = output_tokens[0]
                return processor.decode(gen_ids, skip_special_tokens=True).strip()
            else:
                prompt_str = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True
                )
                inputs = tokenizer(prompt_str, return_tensors="pt", truncation=False).to(model.device)
                with torch.no_grad():
                    output_tokens = model.generate(
                        **inputs,
                        max_new_tokens=max_tokens,
                        do_sample=False,
                        pad_token_id=tokenizer.eos_token_id
                    )
                input_len = inputs.input_ids.shape[1]
                if output_tokens.shape[1] > input_len:
                    gen_ids = output_tokens[0, input_len:]
                else:
                    gen_ids = output_tokens[0]
                return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            return ""
        except Exception as ex:
            return ""

    # Smoke test for Gemma 3 4B (Section 6 of plan)
    if smoke_required:
        print("Running decoder smoke test for Gemma 3 4B...")
        smoke_messages = [{"role": "user", "content": 'Reply with JSON {"action":"final_answer","answer":"ok"}'}]
        smoke_resp = generate_chat_fn(smoke_messages, max_tokens=64)
        print(f"Smoke test response: '{smoke_resp}'")
        if not smoke_resp or not smoke_resp.strip():
            print("WARNING: Gemma 3 4B smoke test emitted empty generation! Marking invalid_empty_generation and skipping.")
            model_statuses[model_name] = "invalid_empty_generation"
            del model
            if tokenizer: del tokenizer
            if processor: del processor
            gc.collect()
            torch.cuda.empty_cache()
            continue

    model_statuses[model_name] = "evaluated"

    # Instantiate the two fair runners
    react_runner = GoldPlanReActRunner(generate_chat_fn)
    isolated_runner = GoldPlanIsolatedRunner(generate_chat_fn)

    runners_map = {
        "goldplan_react": react_runner,
        "goldplan_isolated": isolated_runner
    }

    # Evaluate both modes across all tasks
    for mode_name in ["goldplan_react", "goldplan_isolated"]:
        runner = runners_map[mode_name]
        print(f"\n--- Running {model_name} in {mode_name} mode ({len(BENCHMARK_TASKS)} tasks) ---")
        passed_count = 0
        total_tasks = len(BENCHMARK_TASKS)

        for idx, task in enumerate(BENCHMARK_TASKS):
            res = runner.run_task(task, execute_benchmark_tool, model_name=model_name)
            # Fair scoring of terminal answer
            is_success, score_reason = score_answer(res["final_answer"], task.expected_answer, suite=task.suite)
            res["success"] = is_success
            res["score_reason"] = score_reason

            if is_success:
                passed_count += 1

            all_eval_results.append(res)

            # Write ledger entry immediately
            with open(ledger_path, "a", encoding="utf-8") as f_ledger:
                f_ledger.write(json.dumps(res, ensure_ascii=False) + "\n")

            if (idx + 1) % 10 == 0 or (idx + 1) == total_tasks:
                print(f"  [{idx+1}/{total_tasks}] Passed: {passed_count}/{idx+1} ({passed_count/(idx+1)*100:.1f}%) | Last ans: '{res['final_answer'][:40]}'")

    # Free model memory
    del model
    if tokenizer: del tokenizer
    if processor: del processor
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
# === 8. Summary Reporting & Artifact Manifest ===
import json
import math
import hashlib
from collections import defaultdict
from datetime import datetime, timezone

def wilson_score_interval(k: int, n: int, confidence: float = 0.95):
    if n == 0:
        return 0.0, 0.0
    z = 1.95996  # 95% confidence
    p = k / n
    denominator = 1 + z**2 / n
    centre_adjusted_probability = p + z**2 / (2 * n)
    adjusted_std_error = z * math.sqrt((p * (1 - p) + z**2 / (4 * n)) / n)
    lower = max(0.0, (centre_adjusted_probability - adjusted_std_error) / denominator)
    upper = min(1.0, (centre_adjusted_probability + adjusted_std_error) / denominator)
    return round(lower * 100, 1), round(upper * 100, 1)

print("\n=================== FAIR COMPARISON BENCHMARK REPORT ===================")
# Group results by model, mode
overall = defaultdict(lambda: {"total": 0, "passed": 0, "empty_gen": 0, "invalid_out": 0, "elapsed": 0.0})
tier_stats = defaultdict(lambda: {"total": 0, "passed": 0})
suite_stats = defaultdict(lambda: {"total": 0, "passed": 0})

for r in all_eval_results:
    m = r["model"]
    mode = r["mode"]
    k_ov = (m, mode)
    overall[k_ov]["total"] += 1
    if r["success"]:
        overall[k_ov]["passed"] += 1
    overall[k_ov]["empty_gen"] += r.get("empty_generations", 0)
    overall[k_ov]["invalid_out"] += r.get("invalid_outputs", 0)
    overall[k_ov]["elapsed"] += r.get("elapsed_s", 0.0)

    tier = r.get("tier", "unspecified")
    tier_stats[(m, mode, tier)]["total"] += 1
    if r["success"]:
        tier_stats[(m, mode, tier)]["passed"] += 1

    suite = r.get("suite", "unspecified")
    suite_stats[(m, mode, suite)]["total"] += 1
    if r["success"]:
        suite_stats[(m, mode, suite)]["passed"] += 1

print(f"{'Model':<20} {'Mode':<18} {'Total':<6} {'Passed':<7} {'Pass Rate':<10} {'95% CI':<15} {'EmptyGen':<10} {'InvalidOut':<10}")
print("-" * 105)
for (m, mode), d in sorted(overall.items()):
    n = d["total"]
    k = d["passed"]
    rate = f"{k/n*100:.1f}%" if n else "0%"
    low, high = wilson_score_interval(k, n)
    ci_str = f"[{low}%, {high}%]"
    print(f"{m:<20} {mode:<18} {n:<6} {k:<7} {rate:<10} {ci_str:<15} {d['empty_gen']:<10} {d['invalid_out']:<10}")
print("=" * 105)

# Tier Table
print("\n--- Two-Tier Breakdown (Level 1 Baseline vs Level 2 Advanced) ---")
print(f"{'Model':<20} {'Mode':<18} {'Tier':<10} {'Total':<6} {'Passed':<7} {'Pass Rate':<10}")
print("-" * 75)
for (m, mode, tier), d in sorted(tier_stats.items()):
    n = d["total"]
    k = d["passed"]
    rate = f"{k/n*100:.1f}%" if n else "0%"
    print(f"{m:<20} {mode:<18} {tier:<10} {n:<6} {k:<7} {rate:<10}")

# Suite Table
print("\n--- Suite Breakdown (Synthetic Fixtures) ---")
print(f"{'Model':<20} {'Mode':<18} {'Suite (synthetic)':<22} {'Total':<6} {'Passed':<7} {'Pass Rate':<10}")
print("-" * 85)
for (m, mode, suite), d in sorted(suite_stats.items()):
    n = d["total"]
    k = d["passed"]
    rate = f"{k/n*100:.1f}%" if n else "0%"
    note = " (underpowered)" if n < 8 else ""
    print(f"{m:<20} {mode:<18} {suite + note:<22} {n:<6} {k:<7} {rate:<10}")

# Generate run_manifest.json
manifest = {
    "schema_version": "pdx_fair_run_manifest_v1",
    "benchmark": "PDX Synthetic SLM Dispatch Fair Comparison",
    "conditions": ["goldplan_react", "goldplan_isolated"],
    "engine_invocations": 0,
    "completed_at": datetime.now(timezone.utc).isoformat(),
    "total_evaluations": len(all_eval_results),
    "model_statuses": model_statuses,
    "status": "completed"
}

with open("/kaggle/working/run_manifest.json", "w", encoding="utf-8") as f_m:
    json.dump(manifest, f_m, indent=2, ensure_ascii=False)

# Generate artifact_manifest.json with SHA-256 CAS
artifacts = []
for filename in ["benchmark_ledger.jsonl", "run_manifest.json"]:
    filepath = os.path.join("/kaggle/working", filename)
    if os.path.exists(filepath):
        with open(filepath, "rb") as f_b:
            sha = hashlib.sha256(f_b.read()).hexdigest()
        artifacts.append({"path": filename, "sha256": sha})

artifact_manifest = {
    "schema_version": "pdx_artifact_manifest_v1",
    "artifacts": artifacts
}

with open("/kaggle/working/artifact_manifest.json", "w", encoding="utf-8") as f_am:
    json.dump(artifact_manifest, f_am, indent=2, ensure_ascii=False)

print("\nManifests generated: run_manifest.json & artifact_manifest.json.")
print("Benchmark Run Completed Successfully.")
